# Scribble Evaluation Notebook
Load a scribble from Google Drive, generate conditioned photos,
compute MMD vs 5-class target distribution, and classify via 5-way cosine softmax.

## 1. Setup

In [ ]:
import os, sys, json, gc
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from IPython.display import display
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass('Enter your GitHub personal access token: ')

    repo_url  = f'https://{token}@github.com/orineo1/conditional-matching-paper.git'
    repo_name = 'conditional-matching-paper'
    branch    = 'SD-add-vis-and-table'

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f'/content/{repo_name}'
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

repo_path = f'/content/{repo_name}/SD_cond_SD_controlnet'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

import random

GLOBAL_SEED = 5

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'[Seed] All random seeds set to {seed}')

set_global_seed(GLOBAL_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2. Config & Load Scribble

In [ ]:

CONTROLNET_SCALE = 0.5
N_EVAL           = 100
N_TARGETS        = 300
EVAL_PROMPT      = 'a superrealistic professional photograph of'

# ── 5-class target distribution
TARGET_PROMPTS = [
    ('Woman',            'superrealistic portrait photograph of a woman, extremely feminine features, studio lighting',                                                                          0.25),
    ('Woman with masculine features',      'a superrealistic portrait photograph of a woman with masculine features, heavy brow ridge, studio lighting',                    0.25),
    ('Man with feminine features',  'a superrealistic portrait photograph of a man with extremely feminine feminine features, soft delicate face, high cheekbones, studio lighting',       0.25),
    ('Man',              'a superrealistic portrait photograph of a man, extremely masculine features, studio lighting',                                                                             0.25),
]
assert abs(sum(r for _, _, r in TARGET_PROMPTS) - 1.0) < 1e-6

CLASS_LABELS  = [label  for label, _, _  in TARGET_PROMPTS]
CLASS_PROMPTS = [prompt for _, prompt, _ in TARGET_PROMPTS]
CLASS_FRACS   = [frac   for _, _, frac   in TARGET_PROMPTS]
CLASS_COLORS  = ['crimson', 'orchid', 'slategray', 'steelblue', 'royalblue']

# n images per class in the target
n_per_class = [int(N_TARGETS * f) for f in CLASS_FRACS]

print('Class fractions:')
for label, n in zip(CLASS_LABELS, n_per_class):
    print(f'  {label:<20} n={n}')

## 3. Load Models

In [ ]:
from models     import load_models
from clip_utils import load_clip_model

architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

## 4. Helpers

In [ ]:
from generation    import generate_and_store_cs
from clip_utils    import encode_images_clip
from visualization import plot_row
from metrics       import compute_mmd

def pil_to_tensor(pil_list):
    return torch.cat(
        [TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0
    ).to(next(clip_model.parameters()).device)

def generate_eval_photos(scribble_pil, n=N_EVAL, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = None
    if seed is not None:
        generator = torch.Generator(device=sprinter.device).manual_seed(seed)
    photos = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            result = sprinter(
                prompt=[EVAL_PROMPT] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil',
                generator=generator,
            )
            photos.extend(result.images)
    sprinter.vae.to(dtype=torch.float32)
    return photos

def generate_and_store_cs(pipe, prompt, cond_pil, num_samples, batch_size=2, cn_scale=0.5, seed=None):
    original_vae_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float16)
    all_images, all_lats = [], []

    def latents_callback(p, step_index, timestep, cb_kwargs):
        if step_index == p.num_timesteps - 1:
            p._current_latents = cb_kwargs['latents'].detach().cpu().numpy()
        return cb_kwargs

    generator = None
    if seed is not None:
        generator = torch.Generator(device=pipe.device).manual_seed(seed)
    for i in range(0, num_samples, batch_size):
        curr = min(batch_size, num_samples - i)
        result = pipe(
            prompt=[prompt] * curr,
            image=[cond_pil] * curr,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            callback_on_step_end=latents_callback,
            generator=generator,
        )
        all_images.extend(result.images)
        all_lats.append(pipe._current_latents.reshape(curr, -1))
        print(f'  Progress: {len(all_images)}/{num_samples}', end='\r')
    print()
    pipe.vae.to(dtype=original_vae_dtype)
    return all_images, np.vstack(all_lats)


def encode_text_prompts(prompts):
    clip_model.to(device)
    inputs = clip_processor(
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        # use pooler_output directly from the output we already saw
        outputs = clip_model.text_model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        text_embs = outputs.pooler_output  # [N, D] — this is a plain tensor

    # project through the text projection layer (same as get_text_features does internally)
    text_embs = clip_model.text_projection(text_embs)

    clip_model.to('cpu')
    text_embs = text_embs / text_embs.norm(dim=-1, keepdim=True)
    return text_embs  # [N, D]




def classify_multinomial(image_embs, text_embs):
    """
    5-way cosine softmax classification.
    image_embs: [N, D] (already L2-normalised from encode_images_clip)
    text_embs:  [C, D] (L2-normalised)
    Returns:
        labels  : list of predicted class indices  [N]
        probs   : np.array [N, C]  softmax probabilities
        proportions: np.array [C]  fraction predicted per class
    """
    # ensure both on same device
    image_embs = image_embs.to(device)
    text_embs  = text_embs.to(device)

    image_embs_n = F.normalize(image_embs.float(), dim=-1)
    logits = image_embs_n @ text_embs.T.float()             # [N, C]
    probs  = torch.softmax(logits * 100, dim=-1).cpu().numpy()
    labels = probs.argmax(axis=1).tolist()
    proportions = np.bincount(labels, minlength=len(CLASS_LABELS)) / len(labels)
    return labels, probs, proportions


def multinomial_ci_normal(counts, n_total, z=1.96):
    """Normal-approximation 95% CI for each class proportion."""
    p_hats = counts / n_total
    ses    = np.sqrt(p_hats * (1 - p_hats) / n_total)
    return p_hats, p_hats - z * ses, p_hats + z * ses


print('Helpers ready.')

## 5. Load Best Run Scribble

In [ ]:
# RUNS_ROOT = "/sci/labs/orzuk/ori_m/conditional-matching-paper/SD_cond_SD_controlnet/output"

# best_jid = 44432076

# # 2nd-44432080 - not good as the first mmd 0.374987
# # 3rd - 44432078 - soft-moon-1-need to try - big no no,  VISUAL IMAGES NOT GOOD
# # 4 - 44432074 - amber-spaceship-1 - VISUAL IMAGES NOT GOOD
# # 5 - 44432077 - valiant-snowflake-1 - NO
# # 6 - 44432075 - amber-cherry-5 - maybe
# # 8 - 44432055- mild-sound-1 - NO
# # 9 44432053 - grateful-puddle-1 - current
# # 10 - 44432054 - smooth-cherry-1 - maybe2 - the second after current
# # 11 - 44432076 - ethereal-planet-1 - maybe better then current


# run_dir  = Path(RUNS_ROOT) / f'dps_main_{best_jid}'

# lgd_img         = Image.open(run_dir / 'final_scribble_lgd_cm.png')
# source_scribble = Image.open(run_dir / 'scribble.png')
# source_img      = Image.open(run_dir / 'source_portrait.png')

# fig, axes = plt.subplots(1, 3, figsize=(12, 4))
# for ax, img, title in zip(
#     axes,
#     [source_img, source_scribble, lgd_img],
#     ['Source portrait', 'Source scribble', 'LGD-CM scribble'],
# ):
#     ax.imshow(img); ax.axis('off'); ax.set_title(title)
# plt.tight_layout(); display(fig); plt.close()

# scribble_pil_1 = lgd_img

# # Pre-compute class text embeddings (used throughout)
# class_text_embs = encode_text_prompts(CLASS_PROMPTS)  # [5, D]
# print(f'Class text embeddings: {class_text_embs.shape}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Job to evaluate ───────────────────────────────────────────────────────────
JOB_ID           = 44432053
RUNS_ROOT        = '/content/drive/MyDrive/conditional-matching/runs/InterpolationMenWomen'
run_dir          = Path(RUNS_ROOT) / f'dps_main_{JOB_ID}'

scribble_pil_1         = Image.open(run_dir / 'final_scribble_lgd_cm.png')
source_scribble = Image.open(run_dir / 'scribble.png')

# source_portrait.png may not exist for age runs — skip gracefully
source_portrait_path = run_dir / 'source_portrait.png'
source_img = Image.open(source_portrait_path) if source_portrait_path.exists() else lgd_img

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [source_img, source_scribble,scribble_pil_1],
    ['Source portrait', 'Source scribble (input)', 'LGD-CM scribble'],
):
    ax.imshow(img); ax.axis('off'); ax.set_title(title)
plt.tight_layout(); display(fig); plt.close()

scribble_pil = scribble_pil_1
print(f'Loaded scribble from {run_dir}')

## 6. Build Target Distribution

In [ ]:
print(f'Building target distribution ({N_TARGETS} images, 5 classes)...')
all_target_images = []
with torch.no_grad():
    for i, (label, prompt, frac) in enumerate(TARGET_PROMPTS):
        n_i = n_per_class[i]
        print(f'  [{label}] {n_i} images...')
        imgs, _ = generate_and_store_cs(sprinter, prompt, scribble_pil_1, n_i,
                                        batch_size=2, cn_scale=CONTROLNET_SCALE,
                                        seed=GLOBAL_SEED + i * 1000)
        all_target_images.extend(imgs)
        plot_row(imgs, f'Target {label} ({n_i})', count=min(8, n_i))

with torch.no_grad():
    all_clip_embeddings = encode_images_clip(
        pil_to_tensor(all_target_images), clip_model, clip_processor
    )
print(f'Target CLIP embeddings: {all_clip_embeddings.shape}')

## 7. Generate Eval Photos from Scribble

In [ ]:
print('Generating eval photos from LGD-CM scribble...')
eval_photos_1 = generate_eval_photos(scribble_pil_1, n=N_EVAL, seed=GLOBAL_SEED)
plot_row(eval_photos_1, 'LGD-CM', count=min(10, len(eval_photos_1)))

In [ ]:
# Pick 14 evenly spaced directly from sorted order
picks = [sorted_idx[i * (100 // 14)] for i in range(14)]
picks[2] = sorted_idx[max(0, 2 * (100 // 14) - 1)]  # shift 3rd image left by 5

fig, axes = plt.subplots(2, 7, figsize=(20, 6))
for ax, idx in zip(axes.flatten(), picks):
    ax.imshow(sample_photos[idx])
    ax.axis('off')

axes[0, 0].set_title('← Woman', fontsize=10, color='crimson')
axes[0, 6].set_title('Man →',   fontsize=10, color='steelblue')
fig.suptitle('Gender axis — 14 photos from feminine to masculine', fontsize=12)
plt.tight_layout()
display(fig)
plt.close()

In [ ]:
## 8. PCA on Woman + Man targets, then project everything else

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# ── 1. Encode eval photos ──────────────────────────────────────────────────
with torch.no_grad():
    eval_embs = encode_images_clip(
        pil_to_tensor(eval_photos_1), clip_model, clip_processor
    ).cpu().numpy()

# ── 2. Pull out class embeddings ──────────────────────────────────────────
offsets = np.cumsum([0] + n_per_class)

woman_embs      = all_clip_embeddings[offsets[0]:offsets[1]].cpu().numpy()
woman_masc_embs = all_clip_embeddings[offsets[1]:offsets[2]].cpu().numpy()
man_fem_embs    = all_clip_embeddings[offsets[2]:offsets[3]].cpu().numpy()
man_embs        = all_clip_embeddings[offsets[3]:offsets[4]].cpu().numpy()

# ── 3. Fit PCA on Woman + Man ONLY ────────────────────────────────────────
# old - two extreme  prompts
pca = PCA(n_components=2)
pca.fit(np.vstack([woman_embs, man_embs]))


# ── 4. Project everything onto that PCA space ─────────────────────────────
woman_2d      = pca.transform(woman_embs)
woman_masc_2d = pca.transform(woman_masc_embs)
man_fem_2d    = pca.transform(man_fem_embs)
man_2d        = pca.transform(man_embs)
eval_2d       = pca.transform(eval_embs)

# ── 5. Color by PC1 (orient so Man = positive) ────────────────────────────
flip = -1 if man_2d[:, 0].mean() < woman_2d[:, 0].mean() else 1

all_pc1 = np.concatenate([
    woman_2d[:, 0], woman_masc_2d[:, 0],
    man_fem_2d[:, 0], man_2d[:, 0], eval_2d[:, 0]
]) * flip
vmin, vmax = all_pc1.min(), all_pc1.max()

def score(coords):
    return coords[:, 0] * flip

cmap = cm.RdBu_r  # blue=feminine, red=masculine

In [ ]:
# ── Generate 300 more eval photos and replot PCA ──────────────────────────
print('Generating 300 additional eval photos...')
eval_photos_extra = generate_eval_photos(scribble_pil_1, n=300, seed=GLOBAL_SEED + 123)

with torch.no_grad():
    eval_embs_extra = encode_images_clip(
        pil_to_tensor(eval_photos_extra), clip_model, clip_processor
    ).cpu().numpy()

eval_2d_extra = pca.transform(eval_embs_extra)

fig, ax = plt.subplots(figsize=(16, 5.4))

ax.scatter(woman_2d[:, 0], woman_2d[:, 1],
           c=score(woman_2d), cmap=cmap, vmin=vmin, vmax=vmax,
           s=bg_size, marker='o', alpha=0.6, label='Woman',
           edgecolors='black', linewidths=0.5)
ax.scatter(woman_masc_2d[:, 0], woman_masc_2d[:, 1],
           c=score(woman_masc_2d), cmap=cmap, vmin=vmin, vmax=vmax,
           s=bg_size, marker='o', alpha=0.6, label='Woman w/ masc features',
           edgecolors='black', linewidths=0.5)
ax.scatter(man_fem_2d[:, 0], man_fem_2d[:, 1],
           c=score(man_fem_2d), cmap=cmap, vmin=vmin, vmax=vmax,
           s=bg_size, marker='o', alpha=0.6, label='Man w/ fem features',
           edgecolors='black', linewidths=0.5)
ax.scatter(man_2d[:, 0], man_2d[:, 1],
           c=score(man_2d), cmap=cmap, vmin=vmin, vmax=vmax,
           s=bg_size, marker='o', alpha=0.6, label='Man',
           edgecolors='black', linewidths=0.5)

ax.scatter(eval_2d_extra[:, 0], eval_2d_extra[:, 1],
           c='darkorange', s=500, marker='*', alpha=1.0,
           label='LGD-CM eval photos',
           edgecolors='black', linewidths=1.2, zorder=10)

ax.set_xlabel('PC1 - Man-Woman', fontsize=26, labelpad=15)
ax.set_ylabel('PC2', fontsize=26, labelpad=15)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.tick_params(axis='both', which='both', length=0)
ax.grid(True, linestyle='--', alpha=0.3, color='gray')
ax.set_axisbelow(True)

plt.tight_layout()
display(fig)
plt.close()